# Lab 1.2 · Four indicators from one day

Day 1 · Session 2 (Indicators for latency and quality) · 25 min.

Define first-token time, per-token time, availability and a sampled quality grade on a day of request logs, with the exclusions written down and quality sliced by language with an interval.

**Tasks**
- [ ] Read the request log and its fields · 3 min
- [ ] Write the valid-event filter and its exclusions · 6 min
- [ ] Latency indicators: first token and per token · 6 min
- [ ] Availability as good over valid requests · 4 min
- [ ] Quality grade by language, with a 95% interval · 6 min
- [ ] ✱ Stretch · How many graded samples per day would narrow the Arabic interval to two points?

AI policy: AI-tutor. An assistant may explain the log fields or the interval formula; the exclusions and their reasons are yours.

In [ ]:
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    REPO_URL = "https://github.com/SDAIAAcademy/ae-403.git"
    if not Path("/content/ae-403").exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, "/content/ae-403"], check=True)
    os.chdir("/content/ae-403/labs")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"], check=True)
else:
    while not Path("_tools").exists() and Path.cwd() != Path.cwd().parent:
        os.chdir("..")  # solutions/ notebooks run from labs/
sys.path.insert(0, os.getcwd())
import relia as U
import pandas as pd

## 1 · Read the request log and its fields

One row per request to Athar over one day. `client` is `app`, `web` or `probe` (the uptime
check). `status`: 200 served; 499 the user closed the chat before the answer; 429 rate limited
by the gateway; 500 and 503 server failures. `ttft_ms` and `tpot_ms` exist for served requests.
About 6% of served answers are graded by the quality team (`graded`, `passed`).

In [ ]:
log = U.load("requests_day")
print(log.client.value_counts().to_dict())
print(log.status.value_counts().to_dict())
naive = (log.status == 200).mean()
print(f"naive availability over every row: {naive:.4f}")

## 2 · Write the valid-event filter and its exclusions

An indicator is good events over valid events. ✏️ **Exercise** · Return the valid requests and
a dictionary of exclusions, each with a one-line reason. Decide: is the probe a user? Is a 499
the service's fault? Is a 429 from Athar's own gateway a failure the user felt?

In [ ]:
def valid_requests(log: pd.DataFrame):
    # ✏️ YOUR CODE HERE
    raise NotImplementedError("✏️ Exercise: write your solution here")

In [ ]:
try:
    valid, exclusions = valid_requests(log)
except NotImplementedError:
    exclusions = U.reference("lab_1_2_exclusions")
    valid = log[(log.client != "probe") & (log.status != 499)]
U.keep("lab_1_2_exclusions", exclusions)
for k, v in exclusions.items():
    print(f"excluded {k}: {v}")
U.ok(not (valid.client == "probe").any(), "no probe request is counted as a user")
U.ok((valid.status == 429).any(), "429 stays valid: the user asked and Athar refused")

## 3 · Latency indicators: first token and per token

✏️ **Exercise** · Among valid requests that were served, the share with first token within
1,500 ms, and the share with per-token time within 100 ms. A request that failed has no latency:
it counts in availability, not here.

In [ ]:
def latency_indicators(valid: pd.DataFrame) -> dict:
    # ✏️ YOUR CODE HERE
    raise NotImplementedError("✏️ Exercise: write your solution here")

In [ ]:
try:
    lat = latency_indicators(valid)
except NotImplementedError:
    lat = U.reference("lab_1_2_latency")
U.keep("lab_1_2_latency", lat)
print(lat)
with_probe = (log[log.status == 200].ttft_ms <= 1500).mean()
print(f"the same first-token share with the probe left in: {with_probe:.4f}")

## 4 · Availability as good over valid requests

In [ ]:
def availability(valid: pd.DataFrame) -> float:
    # ✏️ YOUR CODE HERE
    raise NotImplementedError("✏️ Exercise: write your solution here")

In [ ]:
try:
    avail = availability(valid)
except NotImplementedError:
    avail = U.reference("lab_1_2_availability")
U.keep("lab_1_2_availability", avail)
users_and_probe = log[log.status != 499]
print(f"availability {avail:.4f} · naive, cancels counted as failures {naive:.4f} · with the probe counted as a user {(users_and_probe.status == 200).mean():.4f}")
U.ok(naive < avail < (users_and_probe.status == 200).mean(), "each wrong filter moves the number: cancels pull it down, the probe pushes it up")

## 5 · Quality grade by language, with a 95% interval

✏️ **Exercise** · For each language, the pass rate of graded answers and its Wilson interval
(`U.wilson(passed, n)`). A pass rate without its interval invites an argument nobody can settle.

In [ ]:
def quality_by_language(valid: pd.DataFrame) -> pd.DataFrame:
    # ✏️ YOUR CODE HERE
    raise NotImplementedError("✏️ Exercise: write your solution here")

In [ ]:
try:
    quality = quality_by_language(valid)
except NotImplementedError:
    quality = U.reference("lab_1_2_quality")
U.keep("lab_1_2_quality", quality)
print(quality)
ar, en = quality.set_index("lang").loc["ar"], quality.set_index("lang").loc["en"]
U.ok(ar.high < en.rate, "Arabic quality is lower than English, and the interval says it is not noise")

**Look at** the two quality rows. Averaged together they hide a gap that one language's users
feel every day. Every quality objective in this course is sliced by language.

✱ **Stretch** · How many graded samples per day would narrow the Arabic interval to two points?
Try `U.wilson` with larger `n` at the same rate.

## Check yourself

1. Why does leaving the probe in raise the first-token indicator?
   <details><summary>Answer</summary>The probe is one fast request a minute, always within the threshold; it adds good events no user received.</details>
2. Why is a 429 from Athar's own gateway a bad event, while a 499 is excluded?
   <details><summary>Answer</summary>A 429 is Athar refusing a user who wanted an answer; a 499 is the user leaving before the answer was due.</details>
3. What does the Wilson interval add to a pass rate?
   <details><summary>Answer</summary>How far the true rate could be from the sampled one; with small samples, two rates that look different may not be.</details>

**Carry forward** · The four definitions feed Lab 1.3; reference values are in `data/reference/lab_1_2_*.json`.